In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/sandeep-tt/tt-intern-dataset/refs/heads/main/output_file.csv')
df.head()

,date,volume,open,high,low,close,adjclose,ticker
0,2018-06-15,3696200,59.630001,60.080002,59.369999,60.009998,60.009998,TMUS
1,2018-07-26,23518700,43.580002,43.830002,43.220001,43.529999,41.350727,CSCO
2,2018-04-13,2828300,108.849998,109.459999,108.519997,109.260002,101.975182,PEP
3,2018-03-14,21141100,45.340000,45.759998,45.090000,45.279999,42.340096,CSCO
4,2019-05-21,2506000,277.089996,279.779999,274.649994,275.329987,275.329987,AVGO


In [ ]:
def resample_monthly(df):
    df = df.copy()

    df["date"] = pd.to_datetime(df["date"])

    df = df.sort_values("date").set_index("date")

    monthly = df.resample("M").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
        "adjclose": "last"
    })

    return monthly

In [4]:
def calculate_sma(series, window):
    sma = []
    for i in range(len(series)):
        if i + 1 < window:
            sma.append(None)
        else:
            window_sum = series[i + 1 - window:i + 1].sum()
            sma.append(window_sum / window)
    return pd.Series(sma, index=series.index)


In [ ]:
def calculate_ema(series, window):
    multiplier = 2/(window+1)
    ema = [None] * len(series)
    first_ema_index = window-1
    ema[first_ema_index] = series[:window].mean()

    for i in range(first_ema_index+1, len(series)):
        ema[i] = (
            (series.iloc[i] - ema[i-1])*multiplier
            + ema[i-1]
        )

    return pd.Series(ema, index=series.index)


sma and ema could also be calculated by :)

df["SMA_n"] = df["close"].rolling(window=n).mean()

EMA (uses correct smoothing constant internally)

df["EMA_n"] = df["close"].ewm(span=n, adjust=False).mean()

In [ ]:
def add_indicators(df):
    close = df["close"]
    df["SMA_10"] = calculate_sma(close, 10)
    df["SMA_20"] = calculate_sma(close, 20)
    df["EMA_10"] = calculate_ema(close, 10)
    df["EMA_20"] = calculate_ema(close, 20)
    return df


In [ ]:
def write_symbol_file(df, symbol):
    df = df.reset_index()
    df["ticker"] = symbol
    df = df.tail(24)
    df.to_csv(f"result{symbol}.csv", index=False)


In [ ]:
def write_symbol_file(df, symbol):
    df = df.reset_index()
    df["date"] = df["date"].dt.strftime("%m_%Y")
    df = df[[
        "date",
        "open",
        "high",
        "low",
        "close",
        "adjclose",
        "SMA_10",
        "SMA_20",
        "EMA_10",
        "EMA_20"
    ]]
    df = df.tail(24)
    df.to_csv(f"result_{symbol}.csv", index=False)

In [13]:
for symbol, group in df.groupby("ticker"):
    monthly = resample_monthly(group)
    monthly = add_indicators(monthly)
    write_symbol_file(monthly, symbol)

C:\Users\jaisw\AppData\Local\Temp\ipykernel_6628\3322017075.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.resample("M").agg({
C:\Users\jaisw\AppData\Local\Temp\ipykernel_6628\3322017075.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.resample("M").agg({
C:\Users\jaisw\AppData\Local\Temp\ipykernel_6628\3322017075.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.resample("M").agg({
C:\Users\jaisw\AppData\Local\Temp\ipykernel_6628\3322017075.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.resample("M").agg({
C:\Users\jaisw\AppData\Local\Temp\ipykernel_6628\3322017075.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.resample("M").agg({
